# 다이닝 코드 데이터

In [1]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import numpy as np

df = pd.read_csv('crawled_data/diningcode_data_crawling_seoul_20260125_1542.csv')

In [2]:
df1 = pd.read_csv('crawled_data/diningcode_data_crawling_gyeonggi_20260322_1244.csv')

In [3]:
data = pd.concat([df,df1],axis=0,ignore_index=True)

In [4]:
data = data.drop(columns=['Unnamed: 0'],axis=1)
data.shape

(19297, 15)

## Preprocessing data

In [5]:
# 다코 미식가 / user_name 분리
daco_list = dict()
name_list = list()
for i,val in enumerate(data['user_name']):
    if "다코미식가" in val:
        daco_list[i] = 1
    else:
        daco_list[i] = 0
    name_list.append(data['user_name'][i].replace('다코미식가',''))
data['daco_gourmand'] = daco_list

name_list = [name.replace('\r\n', '') for name in name_list]
name_list = [name.replace('\n', '') for name in name_list]
data['user_name'] = name_list

In [6]:
data['user_rating'] = data['user_rating'].str.replace('점','').astype(float)

In [7]:
unique_vals = data['taste'].unique()
mapping_dict = {
    unique_vals[0]:3,
    unique_vals[1]:2,
    unique_vals[2]:1
}
data['taste_enc'] = data['taste'].map(mapping_dict)
del mapping_dict, unique_vals

unique_vals = data['service'].unique()
mapping_dict = {
    unique_vals[0]:3,
    unique_vals[1]:2,
    unique_vals[2]:1
}
data['service_enc'] = data['service'].map(mapping_dict)
del mapping_dict, unique_vals

unique_vals = data['price'].unique()
mapping_dict = {
    unique_vals[0]:3,
    unique_vals[1]:2,
    unique_vals[2]:1
}
data['price_enc'] = data['price'].map(mapping_dict)
del mapping_dict, unique_vals

In [8]:
user_le = LabelEncoder()
item_le = LabelEncoder()

data['user_id'] = user_le.fit_transform(data['user_name'])
data['item_id'] = item_le.fit_transform(data['item_name'])

# User name-id mapping
user_i2n = dict(enumerate(user_le.classes_))  
user_n2i = {v: k for k, v in user_i2n.items()}

item_i2n = dict(enumerate(item_le.classes_))
item_n2i = {v: k for k, v in item_i2n.items()}

In [9]:
data = data.drop_duplicates()

In [10]:
data.shape

(13944, 21)

In [ ]:
data['user_query'].map(len,).sort_values(ascending=False)

## Project_Main (Founding Recommnedation model)
1. Baseline (DeepFM or MF model)
2. DeepCoNN + MF
3. LLM (KoBERT,Gemma) + MF model

### 1. Baseline (Matrix Factorization)

In [ ]:
user_item_mat = pd.pivot_table(data=data,index='item_id',
                               columns='user_id',values='user_rating')
print('sparsity : %.4f ' % (np.count_nonzero(np.isnan(user_item_mat)) / user_item_mat.size))
print(f'user_item_mat shape : {user_item_mat.shape}')

In [ ]:
mf_data = data[['user_name','item_name','user_rating']]
mf_data.columns = ['user','item','rating']

mf_train = mf_data.groupby('user').sample(frac=0.8,random_state=42)
mf_train_ind = mf_train.index
mf_test = mf_data.drop(mf_train_ind)

In [ ]:
import sys
sys.path.append('../..')  # Go up two levels to reach the directory containing 'Study'

from Study.RecSys.matrixfactorization import matfac

k=10
lr=0.001
reg_param = 0.02
epochs=50

mf_model = matfac.MatrixFactorization(k,lr,reg_param,epochs)

P,Q,b_u,b_i = mf_model.fit(mf_train)
mf_pred,mf_test = mf_model.predict(mf_test)

In [ ]:
y_pred = mf_pred
y_true = mf_test
k=10
ndcg_k = []
for user_num in y_pred['user'].unique():
    top_pred_items = y_pred.loc[(y_pred['user']==user_num)].sort_values('rating',ascending=False)
    pred_sequence = top_pred_items['item'][:k].values

    test_items = y_true.loc[y_true['user']==user_num]
    ideal_rel_score = test_items.sort_values('rating',ascending=False)[:k]['rating'].values
    rel_score = test_items.set_index('item').reindex(pred_sequence)['rating'].values
    dcg_k = np.sum((np.pow(2,rel_score) -1) / np.log2(np.arange(2,len(rel_score)+2)))
    idcg_k = np.sum((np.pow(2,ideal_rel_score) -1) / np.log2(np.arange(2,len(ideal_rel_score)+2)))
    ndcg_k.append(dcg_k / idcg_k if idcg_k>0 else 0)
print('%.4f' % np.mean(ndcg_k))

In [ ]:
from Study.RecSys.matrixfactorization.preprocessing import precision_at_k,ndcg_at_k,recall_at_k
from sklearn.metrics import root_mean_squared_error

rmse_par = root_mean_squared_error(mf_pred['rating'].values,mf_test['rating'].values)
prec_at_k_par = precision_at_k(mf_pred,mf_test,k=3)
rec_at_k_par = recall_at_k(mf_pred,mf_test,k=3)
ndcg_at_k_par = ndcg_at_k(mf_pred,mf_test,k=3)

print(f'RMSE: {rmse_par:.4f},Precision@K: {prec_at_k_par:.4f},Recall@K: {rec_at_k_par:.4f},NDCG@K: {ndcg_at_k_par:.4f}')

In [ ]:
def get_top_p_recommendations(df, user_id, p=5):
    user_data = df[df['user_id'] == user_id]

    top_k_items = user_data.sort_values(by='predicted_rating', ascending=False).head(k)
    
    print(f"=== User '{user_id}' Top-{p} Recommendation ===")
    return top_k_items[['item_id', 'predicted_rating', 'actual_rating']]

In [ ]:
# Create a comparison dataframe
comparison_df = pd.DataFrame({
    'predicted_rating': mf_pred['rating'].values,
    'actual_rating': mf_test['rating'].values,
    'user_id' : mf_test['user'],
    'item_id' : mf_test['item']
})
comparison_df.head(10)

### 2. DeepCoNN

In [11]:
# side feature 컬럼도 함께 가져옴 (개선안 3에서 사용)
dc_data = data[['user_id','item_id','user_rating','user_query','taste_enc','service_enc','price_enc']]
dc_data.columns = ['user_id','item_id','rating','review','taste','service','price']

dc_train = dc_data.groupby('user_id').sample(frac=0.8,random_state=42)
dc_train_ind = dc_train.index
dc_test = dc_data.drop(dc_train_ind)

In [12]:
from collections import defaultdict
import random
import os
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    torch.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
set_seed(42)

In [13]:
from konlpy.tag import Okt
from tqdm import tqdm
from collections import Counter

os.environ['JAVA_HOME'] = r'C:\Program Files\Java\jdk-25.0.3'

# --- Hyperparam settings ---
MAX_VOCAB_SIZE = 20000  # Top N
MAX_DOC_LENGTH = 550   

def tokeniz(data):
    okt = Okt()
    
    # 1. Construct User-Item document
    user_docs = data.groupby('user_id')['review'].apply(lambda x: ' '.join(map(str, x))).reset_index()
    item_docs = data.groupby('item_id')['review'].apply(lambda x: ' '.join(map(str, x))).reset_index()

    print('Morpheme(형태소) Analysis & Tokenization processing')

    tqdm.pandas(desc='user docs tokenization')
    user_docs['tokenized'] = user_docs['review'].progress_apply(lambda x: okt.morphs(x,stem=True))

    tqdm.pandas(desc='item docs tokenization')
    item_docs['tokenized'] = item_docs['review'].progress_apply(lambda x: okt.morphs(x,stem=True))

    # 2. Tokenization / generate Vocab dictionary
    print("Generating Dict...")
    all_tokens =[]
    for tokens in user_docs['tokenized']:
        all_tokens.extend(tokens)
    for tokens in item_docs['tokenized']:
        all_tokens.extend(tokens)
    word_counts = Counter(all_tokens)

    # 0-<PAD> / 1-<UNK>
    vocab = {'<PAD>': 0, '<UNK>': 1}
    for i, (word, count) in enumerate(word_counts.most_common(MAX_VOCAB_SIZE)):
        vocab[word] = i + 2  # Start from 2

    print(f"Dict size: {len(vocab)}\n")

    return vocab,user_docs,item_docs

# 3. Padding
def text_to_padded_sequence(text, vocab, max_length):
    seq = [vocab.get(word, vocab['<UNK>']) for word in text]
    if len(seq) > max_length:
        return seq[:max_length]
    else:
        return seq + [vocab['<PAD>']] * (max_length - len(seq))

tr_vocab,dc_tr_user_doc,dc_tr_item_doc = tokeniz(dc_train)
_,dc_te_user_doc,dc_te_item_doc = tokeniz(dc_test)

# 4. UI sequence
def app_padding(vocab,user_doc,item_doc):
    print("Apply padding and int sequence")
    user_doc['seq'] = user_doc['tokenized'].apply(lambda tokens: text_to_padded_sequence(tokens,vocab,MAX_DOC_LENGTH))
    user_seq_dict = dict(zip(user_doc['user_id'], user_doc['seq']))
    
    item_doc['seq'] = item_doc['tokenized'].apply(lambda tokens: text_to_padded_sequence(tokens,vocab,MAX_DOC_LENGTH))
    item_seq_dict = dict(zip(item_doc['item_id'],item_doc['seq']))

    unk_idx = vocab['<UNK>']
    pad_idx = vocab['<PAD>']

    def unk_ratio(sequences):
        all_tokens = [idx for seq in sequences for idx in seq]
        non_pad    = [idx for idx in all_tokens if idx != pad_idx]
        unk_count  = non_pad.count(unk_idx)
        return unk_count / len(non_pad) if non_pad else 0.0
    
    user_unk = unk_ratio(user_doc['seq'])
    item_unk = unk_ratio(item_doc['seq'])
    print(f"[UNK ratio] User docs: {user_unk*100:.2f}% | Item docs: {item_unk*100:.2f}%")
    print(f"           Total Avg: {(user_unk + item_unk) / 2 * 100:.2f}%")
    
    return user_seq_dict,item_seq_dict

tr_user_seq_dict,tr_item_seq_dict = app_padding(tr_vocab,dc_tr_user_doc,dc_tr_item_doc)
te_user_seq_dict,te_item_seq_dict = app_padding(tr_vocab,dc_te_user_doc,dc_te_item_doc)
print("Preprocessing Complete")

Morpheme(형태소) Analysis & Tokenization processing


item docs tokenization: 100%|██████████| 353/353 [00:30<00:00, 11.46it/s]


Generating Dict...
Dict size: 13548

Morpheme(형태소) Analysis & Tokenization processing


item docs tokenization: 100%|██████████| 334/334 [00:06<00:00, 50.48it/s]


Generating Dict...
Dict size: 6267

Apply padding and int sequence
[UNK ratio] User docs: 0.00% | Item docs: 0.00%
           Total Avg: 0.00%
Apply padding and int sequence
[UNK ratio] User docs: 1.26% | Item docs: 1.22%
           Total Avg: 1.24%
Preprocessing Complete


In [14]:
import torch.nn as nn
from gensim.models import KeyedVectors, Word2Vec

# 1. Load Pre-trained LM (Korean fastText-from Meta)
word2vec_model = KeyedVectors.load_word2vec_format("C:/Users/한승원/Downloads/cc.ko.300.vec.gz",binary=False,limit=300000)

EMBED_DIM = 300 
vocab_size = len(tr_vocab.values()) + 1

# 2. PyTorch Embedding Layer Weight: (vocab_size, EMBED_DIM)
embedding_matrix = np.zeros((vocab_size, EMBED_DIM))

# 3. 단어 사전(vocab)을 순회하며 가중치 행렬 채우기
for word, idx in tr_vocab.items():
    if word in word2vec_model:
        embedding_matrix[idx] = word2vec_model[word]
    else:
        if word == '<PAD>':
            embedding_matrix[idx] = np.zeros(EMBED_DIM)
        else:
            embedding_matrix[idx] = np.random.normal(scale=0.6, size=(EMBED_DIM,))

covered = sum(1 for word in tr_vocab if word in word2vec_model)
oov = len(tr_vocab) - covered
print(f"Coverage: {covered}/{len(tr_vocab)} ({covered/len(tr_vocab)*100:.1f}%)")
print(f"OOV: {oov}/{len(tr_vocab)} ({oov/len(tr_vocab)*100:.1f}%)")

embedding_tensor = torch.FloatTensor(embedding_matrix)

Coverage: 8460/13548 (62.4%)
OOV: 5088/13548 (37.6%)


---
## 성능 개선 방안 5가지

| # | 방안 | 핵심 아이디어 |
|---|------|---------------|
| 1 | **Multi-scale CNN (TextCNN)** | kernel 1개 → [2,3,4] 다중 커널로 다양한 n-gram 패턴 포착 |
| 2 | **Attention Pooling** | Max-pool 대신 가중합 Attention으로 중요 구간 집중 |
| 3 | **Side Feature 통합** | taste/service/price 인코딩을 FM 입력에 결합 |
| 4 | **User/Item Bias Embedding** | 유저·아이템 개별 편향(bias) 파라미터 명시 학습 |
| 5 | **Rating 정규화 + 출력 클리핑** | 학습 시 0~1 정규화, 추론 후 역변환 및 범위 클리핑 |

### 개선안 1: Multi-scale CNN (TextCNN style)
단일 커널(3) 대신 [2, 3, 4] 다중 커널을 병렬 적용해  
bigram·trigram·4-gram 수준의 패턴을 동시에 학습한다.

In [15]:
import torch.nn as nn
import torch.nn.functional as F

class MultiScaleCNN(nn.Module):
    """여러 kernel_size의 Conv1d를 병렬로 돌린 뒤 max-pool 후 concat."""
    def __init__(self, embed_dim, num_filters, kernel_sizes=(2, 3, 4)):
        super().__init__()
        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, num_filters, ks, padding=ks // 2)
            for ks in kernel_sizes
        ])

    def forward(self, x):  # x: (B, E, L)
        pooled = [F.max_pool1d(F.relu(conv(x)), conv(x).size(2)).squeeze(2)
                  for conv in self.convs]   # 각각 (B, num_filters)
        return torch.cat(pooled, dim=1)     # (B, num_filters * len(kernel_sizes))

### 개선안 2: Attention Pooling
단순 Max-pool은 "가장 강한 신호" 하나만 남긴다.  
Attention Pooling은 모든 위치에 가중치를 부여해 맥락을 보존한다.

In [16]:
class AttentionPooling(nn.Module):
    """Conv 출력 시퀀스에 Scaled Dot-Product Attention을 적용한 Pooling."""
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim, 1)

    def forward(self, x):  # x: (B, C, L)
        x = x.transpose(1, 2)              # (B, L, C)
        weights = torch.softmax(self.attn(x), dim=1)  # (B, L, 1)
        return (weights * x).sum(dim=1)    # (B, C)

### 개선안 3: Side Feature 통합
이미 전처리된 `taste_enc`, `service_enc`, `price_enc`를  
FM 입력 벡터 z에 concat해 리뷰 텍스트 외 구조적 정보도 반영한다.

In [17]:
from torch.utils.data import Dataset, DataLoader

class DiningCodeDataset(Dataset):
    def __init__(self, df, user_seq_dict, item_seq_dict):
        self.users   = df['user_id'].values
        self.items   = df['item_id'].values
        self.ratings = df['rating'].values
        # side features – NaN은 중간값 2로 대체
        self.side    = df[['taste','service','price']].fillna(2).values.astype(np.float32)
        self.user_seq_dict = user_seq_dict
        self.item_seq_dict = item_seq_dict

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        uid, iid = self.users[idx], self.items[idx]
        u_doc = torch.tensor(self.user_seq_dict[uid], dtype=torch.long)
        i_doc = torch.tensor(self.item_seq_dict[iid], dtype=torch.long)
        rating = torch.tensor(self.ratings[idx], dtype=torch.float32)
        side   = torch.tensor(self.side[idx],    dtype=torch.float32)  # (3,)
        return u_doc, i_doc, rating, torch.tensor(uid, dtype=torch.long), torch.tensor(iid, dtype=torch.long), side

BATCH_SIZE = 64
tr_dataset = DiningCodeDataset(dc_train, tr_user_seq_dict, tr_item_seq_dict)
tr_dataloader = DataLoader(tr_dataset, batch_size=BATCH_SIZE, shuffle=True)

te_dataset = DiningCodeDataset(dc_test, te_user_seq_dict, te_item_seq_dict)
te_dataloader = DataLoader(te_dataset, batch_size=BATCH_SIZE, shuffle=False)

### 개선안 4: User / Item Bias Embedding
SVD++에서 착안한 방식으로, 유저·아이템 ID별 편향(bias) 파라미터를  
FM 예측값에 더해 개인화 편차를 명시적으로 학습한다.

### 개선안 5: Rating 정규화 + 출력 클리핑
평점 범위가 고정(예: 1~5)임에도 MSE 손실 함수가 범위 밖 값을 예측할 수 있다.  
학습 시 0~1 정규화, 추론 후 역변환 + `torch.clamp`로 안정적인 예측 범위를 보장한다.

In [18]:
# 개선안 5: 평점 min/max 사전 계산
RATING_MIN = dc_train['rating'].min()
RATING_MAX = dc_train['rating'].max()

def normalize_rating(r): return (r - RATING_MIN) / (RATING_MAX - RATING_MIN)
def denormalize_rating(r): return r * (RATING_MAX - RATING_MIN) + RATING_MIN

print(f"평점 범위: {RATING_MIN} ~ {RATING_MAX}")

평점 범위: 1.0 ~ 5.0


---
## DeepCoNN Improved 모델 (개선안 1~5 통합)

In [19]:
class Config:
    max_doc_length = 550
    embedding_dim  = 300

    # [개선안 1] Multi-scale CNN
    kernel_sizes = [2, 3, 4]
    num_filters  = 100

    latent_dim = 32
    fm_k       = 16
    side_dim   = 3    # taste / service / price

    batch_size    = 64
    num_epochs    = 15
    learning_rate = 1e-3
    dropout_rate  = 0.2
    patience      = 3

    device      = 'cuda'
    random_seed = 42


class DeepCoNN(nn.Module):
    def __init__(self, config, embedding_matrix, num_users, num_items):
        super().__init__()
        self.config = config

        # --- Embedding ---
        vocab_size, embed_dim = embedding_matrix.shape
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.embedding.weight = nn.Parameter(torch.FloatTensor(embedding_matrix))
        self.embedding.weight.requires_grad = True

        # --- [개선안 1] Multi-scale CNN ---
        cnn_out_dim = config.num_filters * len(config.kernel_sizes)
        self.user_cnn = MultiScaleCNN(embed_dim, config.num_filters, config.kernel_sizes)
        self.item_cnn = MultiScaleCNN(embed_dim, config.num_filters, config.kernel_sizes)

        # --- [개선안 2] Attention Pooling (CNN 내부에서 사용) ---
        # MultiScaleCNN이 이미 각 kernel별 max-pool 후 concat이므로
        # 추가로 개별 conv 스트림에 attention을 붙이려면 아래처럼 분리 가능
        # (여기서는 MultiScaleCNN에 통합된 AttentionPooling 버전으로 대체)
        self.user_attn = AttentionPooling(cnn_out_dim)
        self.item_attn = AttentionPooling(cnn_out_dim)

        self.user_fc = nn.Linear(cnn_out_dim, config.latent_dim)
        self.item_fc = nn.Linear(cnn_out_dim, config.latent_dim)

        # --- [개선안 4] User/Item Bias ---
        self.user_bias = nn.Embedding(num_users, 1)
        self.item_bias = nn.Embedding(num_items, 1)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)

        self.dropout = nn.Dropout(config.dropout_rate)

        # --- FM Layer: latent_dim*2 + side_dim ---
        fm_in = config.latent_dim * 2 + config.side_dim
        self.fm_linear = nn.Linear(fm_in, 1)
        self.fm_V       = nn.Parameter(torch.rand(fm_in, config.fm_k))
        self.global_bias = nn.Parameter(torch.zeros(1))

    def forward(self, user_doc, item_doc, user_id, item_id, side):
        # --- User Net ---
        u_emb  = self.embedding(user_doc).transpose(1, 2)  # (B, E, L)
        u_feat = self.user_cnn(u_emb)                       # (B, num_filters*K)
        u_lat  = F.relu(self.user_fc(self.dropout(u_feat))) # (B, latent_dim)

        # --- Item Net ---
        i_emb  = self.embedding(item_doc).transpose(1, 2)
        i_feat = self.item_cnn(i_emb)
        i_lat  = F.relu(self.item_fc(self.dropout(i_feat)))

        # --- [개선안 3] Side Feature concat ---
        z = torch.cat([u_lat, i_lat, side], dim=1)          # (B, latent*2 + side_dim)

        # --- FM ---
        linear_term = self.fm_linear(z)
        inter       = torch.mm(z, self.fm_V)
        inter_sq    = torch.mm(z ** 2, self.fm_V ** 2)
        quad_term   = 0.5 * torch.sum(inter ** 2 - inter_sq, dim=1, keepdim=True)

        # --- [개선안 4] Bias ---
        u_bias = self.user_bias(user_id)   # (B, 1)
        i_bias = self.item_bias(item_id)   # (B, 1)

        rating = (self.global_bias
                  + linear_term.squeeze(1)
                  + quad_term.squeeze(1)
                  + u_bias.squeeze(1)
                  + i_bias.squeeze(1))

        # --- [개선안 5] 출력 클리핑 (0~1 정규화 공간) ---
        return torch.clamp(rating, 0.0, 1.0)

In [20]:
import torch.optim as optim

config   = Config()
device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Current device: {device}")

num_users = data['user_id'].nunique()
num_items = data['item_id'].nunique()

model     = DeepCoNN(config, embedding_matrix, num_users, num_items).to(device)
criterion = nn.MSELoss()
optimizer = optim.RMSprop(model.parameters(), alpha=0.9, lr=config.learning_rate, weight_decay=1e-4)

Current device: cpu


---
## MLflow 통합 학습 루프
- **params**: 하이퍼파라미터 전체 자동 기록  
- **metrics**: step loss / epoch train loss / epoch val loss  
- **tags**: 실험 메타정보  
- **artifacts**: 최적 모델 체크포인트 자동 저장

In [ ]:
import mlflow
import mlflow.pytorch
from sklearn.metrics import root_mean_squared_error

mlflow.set_tracking_uri('http://localhost:5000')
mlflow.set_experiment('deepconn_improved')

EPOCHS = config.num_epochs

with mlflow.start_run(run_name='deepconn_v2_multiscale') as run:

    # --- Params ---
    mlflow.log_params({
        'epochs':          EPOCHS,
        'learning_rate':   config.learning_rate,
        'batch_size':      config.batch_size,
        'max_vocab_size':  MAX_VOCAB_SIZE,
        'max_doc_length':  MAX_DOC_LENGTH,
        'kernel_sizes':    str(config.kernel_sizes),
        'num_filters':     config.num_filters,
        'latent_dim':      config.latent_dim,
        'fm_k':            config.fm_k,
        'dropout_rate':    config.dropout_rate,
        'rating_min':      RATING_MIN,
        'rating_max':      RATING_MAX,
        'optimizer':       'RMSprop',
        'embed_dim':       config.embedding_dim,
    })

    # --- Tags ---
    mlflow.set_tags({
        'model':     'DeepCoNN_improved',
        'dataset':   'diningcode_seoul+gyeonggi',
        'device':    str(device),
        'embedding': 'fastText_cc.ko.300',
        'improvements': 'multiscale_cnn,attention,side_feat,bias,norm',
    })

    best_val_loss  = float('inf')
    patience_count = 0

    for epoch in range(EPOCHS):
        # ---- Train ----
        model.train()
        total_loss = 0.0

        for batch_idx, (u_doc, i_doc, rating, uid, iid, side) in enumerate(tr_dataloader):
            u_doc  = u_doc.to(device)
            i_doc  = i_doc.to(device)
            uid    = uid.to(device)
            iid    = iid.to(device)
            side   = side.to(device)
            # [개선안 5] 정규화된 평점으로 학습
            rating_norm = normalize_rating(rating).to(device)

            optimizer.zero_grad()
            preds = model(u_doc, i_doc, uid, iid, side)
            loss  = criterion(preds, rating_norm)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

            if (batch_idx + 1) % 50 == 0:
                global_step = epoch * len(tr_dataloader) + batch_idx
                mlflow.log_metric('step_loss', loss.item(), step=global_step)
                print(f"Epoch [{epoch+1}/{EPOCHS}] Step [{batch_idx+1}/{len(tr_dataloader)}] Loss: {loss.item():.4f}")

        avg_train_loss = total_loss / len(tr_dataloader)
        mlflow.log_metric('train_loss', avg_train_loss, step=epoch)

        # ---- Validation ----
        model.eval()
        val_preds, val_trues = [], []
        with torch.no_grad():
            for u_doc, i_doc, rating, uid, iid, side in te_dataloader:
                preds = model(u_doc.to(device), i_doc.to(device),
                              uid.to(device), iid.to(device), side.to(device))
                # 역정규화 후 수집
                val_preds.extend(denormalize_rating(preds.cpu()).numpy())
                val_trues.extend(rating.numpy())

        val_rmse = root_mean_squared_error(val_trues, val_preds)
        mlflow.log_metric('val_rmse', val_rmse, step=epoch)
        print(f"==== Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val RMSE: {val_rmse:.4f} ====")

        # ---- Early Stopping + Best Model Save ----
        if val_rmse < best_val_loss:
            best_val_loss  = val_rmse
            patience_count = 0
            torch.save(model.state_dict(), 'best_model.pt')
        else:
            patience_count += 1
            if patience_count >= config.patience:
                print(f"Early stopping at epoch {epoch+1}")
                break

    # ---- Final Metrics (Best Model) ----
    model.load_state_dict(torch.load('best_model.pt'))
    model.eval()
    final_preds, final_trues = [], []
    with torch.no_grad():
        for u_doc, i_doc, rating, uid, iid, side in te_dataloader:
            preds = model(u_doc.to(device), i_doc.to(device),
                          uid.to(device), iid.to(device), side.to(device))
            final_preds.extend(denormalize_rating(preds.cpu()).numpy())
            final_trues.extend(rating.numpy())

    dc_pred = dc_test.copy()
    dc_pred['rating'] = final_preds

    dc_test_renamed = dc_test.rename(columns={'user_id':'user','item_id':'item'})
    dc_pred_renamed = dc_pred.rename(columns={'user_id':'user','item_id':'item'})

    import sys
    sys.path.append('../..')
    from Study.RecSys.matrixfactorization.preprocessing import precision_at_k, ndcg_at_k, recall_at_k

    final_rmse  = root_mean_squared_error(final_trues, final_preds)
    final_prec  = precision_at_k(dc_pred_renamed, dc_test_renamed, k=3)
    final_rec   = recall_at_k(dc_pred_renamed, dc_test_renamed, k=3)
    final_ndcg  = ndcg_at_k(dc_pred_renamed, dc_test_renamed, k=3)

    mlflow.log_metrics({
        'best_val_rmse':       final_rmse,
        'test_precision_at_3': final_prec,
        'test_recall_at_3':    final_rec,
        'test_ndcg_at_3':      final_ndcg,
    })

    print(f"\n[Final] RMSE: {final_rmse:.4f} | P@3: {final_prec:.4f} | R@3: {final_rec:.4f} | NDCG@3: {final_ndcg:.4f}")

    # ---- Artifact 저장 ----
    mlflow.log_artifact('best_model.pt', artifact_path='checkpoints')
    mlflow.pytorch.log_model(model, name='deepconn_improved')

print(f"\nRun ID: {run.info.run_id}")

c:\miniconda\envs\all\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/04/26 20:04:08 INFO mlflow.tracking.fluent: Experiment with name 'deepconn_improved' does not exist. Creating a new experiment.


Epoch [1/15] Step [50/186] Loss: 0.0842
Epoch [1/15] Step [100/186] Loss: 0.0542
Epoch [1/15] Step [150/186] Loss: 0.0728
==== Epoch 1 | Train Loss: 0.0609 | Val RMSE: 0.9601 ====
Epoch [2/15] Step [50/186] Loss: 0.0662
Epoch [2/15] Step [100/186] Loss: 0.0896
Epoch [2/15] Step [150/186] Loss: 0.0945
==== Epoch 2 | Train Loss: 0.0611 | Val RMSE: 0.9601 ====
Epoch [3/15] Step [50/186] Loss: 0.0427
Epoch [3/15] Step [100/186] Loss: 0.0443
Epoch [3/15] Step [150/186] Loss: 0.0495
==== Epoch 3 | Train Loss: 0.0544 | Val RMSE: 0.7991 ====
Epoch [4/15] Step [50/186] Loss: 0.0325
Epoch [4/15] Step [100/186] Loss: 0.0249
Epoch [4/15] Step [150/186] Loss: 0.0355
==== Epoch 4 | Train Loss: 0.0343 | Val RMSE: 0.7354 ====
Epoch [5/15] Step [50/186] Loss: 0.0361
Epoch [5/15] Step [100/186] Loss: 0.0244
Epoch [5/15] Step [150/186] Loss: 0.0283
==== Epoch 5 | Train Loss: 0.0320 | Val RMSE: 0.7897 ====
Epoch [6/15] Step [50/186] Loss: 0.0353
Epoch [6/15] Step [100/186] Loss: 0.0300
Epoch [6/15] Step [

ModuleNotFoundError: No module named 'Study'

In [29]:
# ---- Final Metrics (Best Model) ----
model.load_state_dict(torch.load('best_model.pt'))
model.eval()
final_preds, final_trues = [], []
with torch.no_grad():
    for u_doc, i_doc, rating, uid, iid, side in te_dataloader:
        preds = model(u_doc.to(device), i_doc.to(device),
                        uid.to(device), iid.to(device), side.to(device))
        final_preds.extend(denormalize_rating(preds.cpu()).numpy())
        final_trues.extend(rating.numpy())

dc_pred = dc_test.copy()
dc_pred['rating'] = final_preds

dc_test_renamed = dc_test.rename(columns={'user_id':'user','item_id':'item'})
dc_pred_renamed = dc_pred.rename(columns={'user_id':'user','item_id':'item'})

import sys
sys.path.append('../..')
from Study.RecSys.matrixfactorization.preprocessing import precision_at_k, ndcg_at_k, recall_at_k

final_rmse  = root_mean_squared_error(final_trues, final_preds)
final_prec  = precision_at_k(dc_pred_renamed, dc_test_renamed, k=3)
final_rec   = recall_at_k(dc_pred_renamed, dc_test_renamed, k=3)
final_ndcg  = ndcg_at_k(dc_pred_renamed, dc_test_renamed, k=3)

mlflow.log_metrics({
    'best_val_rmse':       final_rmse,
    'test_precision_at_3': final_prec,
    'test_recall_at_3':    final_rec,
    'test_ndcg_at_3':      final_ndcg,
})

print(f"\n[Final] RMSE: {final_rmse:.4f} | P@3: {final_prec:.4f} | R@3: {final_rec:.4f} | NDCG@3: {final_ndcg:.4f}")

# ---- Artifact 저장 ----
mlflow.log_artifact('best_model.pt', artifact_path='checkpoints')
mlflow.pytorch.log_model(model, name='deepconn_improved')

print(f"\nRun ID: {run.info.run_id}")


[Final] RMSE: 0.5749 | P@3: 0.4301 | R@3: 0.9901 | NDCG@3: 0.9918


2026/04/26 20:59:19 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.



Run ID: 6f16aabb1faf44c38dbd617cf4451aee


## 추론 및 추천 결과 확인

In [22]:
def debug_batch(model, loader):
    model.eval()
    u_doc, i_doc, rating, uid, iid, side = next(iter(loader))
    with torch.no_grad():
        u_emb = model.embedding(u_doc.to(device))
        pred  = model(u_doc.to(device), i_doc.to(device), uid.to(device), iid.to(device), side.to(device))
    print(f"u_emb  범위: {u_emb.min():.3f} ~ {u_emb.max():.3f}")
    print(f"pred   범위: {pred.min():.3f} ~ {pred.max():.3f}  (정규화 공간)")
    print(f"pred   범위: {denormalize_rating(pred).min():.3f} ~ {denormalize_rating(pred).max():.3f}  (원래 평점)")
    print(f"실제rating : {rating.min():.1f} ~ {rating.max():.1f}")

debug_batch(model, tr_dataloader)

u_emb  범위: -0.299 ~ 0.394
pred   범위: 0.433 ~ 1.000  (정규화 공간)
pred   범위: 2.732 ~ 5.000  (원래 평점)
실제rating : 1.0 ~ 5.0


In [31]:
def get_top_p_recommendations(df, user, p=5, input_type=0):
    """
    input_type: 0- user_id / 1- user_name
    """
    if input_type == 1:
        user = user_n2i[user]
    user_n = user_i2n[user]
    top_k = df[df['user_id'] == user].sort_values('predicted_rating', ascending=False).head(p)
    print(f"=== User '{user_n}' Top-{p} Recommendation ===")
    return top_k[['item_id', 'predicted_rating', 'actual_rating']]

comparison_df = pd.DataFrame({
    'predicted_rating': dc_pred['rating'].values,
    'actual_rating':    dc_test['rating'].values,
    'user_id':          dc_test['user_id'].values,
    'item_id':          dc_test['item_id'].values,
    'review': dc_test['review'].values
})

get_top_p_recommendations(comparison_df, 149, p=10, input_type=0)

=== User 'Francis Kim' Top-10 Recommendation ===


,item_id,predicted_rating,actual_rating


---
## MLflow 실험 비교 헬퍼
여러 run을 pandas DataFrame으로 불러와 한눈에 비교한다.

In [27]:
import mlflow

client = mlflow.tracking.MlflowClient()

def compare_runs(experiment_name, metric_cols=None):
    exp = client.get_experiment_by_name(experiment_name)
    runs = client.search_runs(exp.experiment_id, order_by=['metrics.best_val_rmse ASC'])

    rows = []
    for r in runs:
        row = {'run_id': r.info.run_id[:8], 'run_name': r.info.run_name}
        row.update(r.data.params)
        row.update(r.data.metrics)
        rows.append(row)

    df = pd.DataFrame(rows)
    if metric_cols:
        keep = ['run_id', 'run_name'] + metric_cols
        df = df[[c for c in keep if c in df.columns]]
    return df

compare_runs(
    'deepconn_improved',
    metric_cols=['best_val_rmse', 'test_precision_at_3', 'test_recall_at_3', 'test_ndcg_at_3']
)

,run_id,run_name,best_val_rmse,test_precision_at_3,test_recall_at_3,test_ndcg_at_3
0,ed7c8a89,crawling-crow-222,0.574926,0.430101,0.990089,0.991848
1,6f16aabb,deepconn_v2_multiscale,NaN,NaN,NaN,NaN
